## Просто Catboost

In [4]:
import catboost as cb

3. Как работает обработка категориальных признаков:
Target Encoding (Ordered Target Statistics)

Вместо one-hot encoding CatBoost использует:
1. Для регрессии: среднее целевой переменной по категории
2. Для классификации: вероятность класса по категории
3. Добавляет шум для предотвращения переобучения

Пример: категория "город"
1. Москва: средний таргет = 0.65
2. СПб: средний таргет = 0.72
3. Казань: средний таргет = 0.58

Обычный градиентный бустинг:
1. Считает градиенты на всей выборке
2. Строит дерево
3. Проблема: переобучение (target leakage)

Ordered Boosting в CatBoost:
1. Для каждого объекта использует только предыдущие объекты
2. Исключает target leakage
3. Защита от переобучения

In [10]:
import catboost as cb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Создание данных с категориальными признаками
data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 50],
    'city': ['Moscow', 'SPb', 'Moscow', 'Kazan', 'SPb', 'Moscow'],  # Категориальный
    'income': [50000, 60000, 55000, 70000, 65000, 80000],
    'education': ['high', 'medium', 'high', 'low', 'medium', 'high'],  # Категориальный
    'target': [1, 0, 1, 0, 1, 0]
})

# Разделение на признаки и целевую переменную
X = data.drop('target', axis=1)
y = data['target']

# Указание категориальных признаков
cat_features = ['city', 'education']

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Создание модели
model = cb.CatBoostClassifier(
    iterations=1000,  # Количество деревьев
    learning_rate=0.1,
    depth=6,  # Глубина деревьев
    loss_function='Logloss',  # Функция потерь
    eval_metric='AUC',  # Метрика для валидации
    cat_features=cat_features,  # Категориальные признаки
    random_seed=42,
    verbose=100,  # Вывод каждые 100 итераций
    early_stopping_rounds=50  # Ранняя остановка
)

# Обучение с валидацией
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    plot=True  # Построение графиков обучения
)

# Предсказание
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Оценка
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.5000000	best: 0.5000000 (0)	total: 57.1ms	remaining: 57.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1
bestIteration = 7

Shrink model to first 8 iterations.
Accuracy: 0.5000
ROC-AUC: 1.0000


In [ ]:
model = cb.CatBoostClassifier(
    # Основные параметры
    iterations=1000,  # Количество деревьев
    learning_rate=0.03,  # Скорость обучения (лучше 0.03-0.1)
    depth=6,  # Глубина деревьев (6-10)
    
    # Регуляризация
    l2_leaf_reg=3,  # L2 регуляризация
    random_strength=1,  # Сила случайности
    bagging_temperature=1,  # Температура бэггинга
    
    # Работа с категориями
    one_hot_max_size=2,  # One-hot для категорий с <= 2 уникальных значений
    has_time=False,  # Есть ли временной порядок
    
    # Контроль переобучения
    early_stopping_rounds=50,
    use_best_model=True,
    
    # Производительность
    thread_count=-1,  # Все ядра
    task_type='CPU',  # 'CPU' или 'GPU'
    
    # Другие
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

In [ ]:
# Gridsearch with CatBoost

from sklearn.model_selection import GridSearchCV

# Определение параметров для поиска
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [500, 1000]}

# Создание модели
model = cb.CatBoostClassifier(
    cat_features=cat_features,
    random_seed=42,
    verbose=0)

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1)

grid_search.fit(X_train, y_train)
print(f"Лучшие параметры: {grid_search.best_params_}")

In [ ]:
# Практические советы:

# 1. Всегда указывайте cat_features
# 2. Используйте early_stopping_rounds
# 3. Начинайте с learning_rate=0.03-0.05
# 4. Для GPU: depth=8-10, для CPU: depth=6-8
# 5. Используйте Pool для больших данных
# 6. Для текстовых категорий используйте TextFeatures

# Пример с текстовыми признаками
model = cb.CatBoostClassifier(
    text_features=['description', 'title'],  # Текстовые колонки
    tokenizers=[{'tokenizer_id': 'Sense', 'delimiter': ' ', 'lowercasing': 'true'}],
    dictionaries=[{'dictionary_id': 'Word', 'gram_order': '1'}],
    feature_calcers=['BoW', 'NaiveBayes'],
    iterations=1000)

## Optuna with CatBoost

In [ ]:
import optuna
import catboost as cb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 50],
    'city': ['Moscow', 'SPb', 'Moscow', 'Kazan', 'SPb', 'Moscow'],
    'income': [50000, 60000, 55000, 70000, 65000, 80000],
    'education': ['high', 'medium', 'high', 'low', 'medium', 'high'],
    'target': [1, 0, 1, 0, 1, 0]
})

X = data.drop('target', axis=1)
y = data['target']

cat_features = ['city', 'education']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
def objective(trial):

    params = {
        # Основные
        'iterations': trial.suggest_int('iterations', 300, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),

        # Регуляризация
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),

        # Метрики и цель
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',

        # Служебные
        'random_seed': 42,
        'verbose': 0,
        'thread_count': -1
    }

    model = cb.CatBoostClassifier(**params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=50,
        use_best_model=True
    )

    y_pred_proba = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_pred_proba)

    return auc

In [ ]:
# запуск Optuna 

study = optuna.create_study(
    direction='maximize',
    study_name='catboost_auc_no_pool'
)

study.optimize(objective, n_trials=50)

print('Best AUC:', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
    
# final model 

best_model = cb.CatBoostClassifier(
    **study.best_params,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

best_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

In [ ]:
# 1. cat_features ОБЯЗАТЕЛЬНО передавать в fit()
# 2. eval_set должен быть тем же форматом, что и train
# 3. Строковые категории — ок
# 4. Text / weights / ranking — без Pool уже не получится
# 5. Для Optuna такой вариант полностью валиден

In [1]:
import pandas as pd 

df = pd.read_csv('/Users/user/Downloads/BACI_HS22_Y2024_V202601.csv')

df1 = df.sample(n=10000, random_state =42)
df1.to_csv('artem_gay.csv')

In [1]:
%pip list

Package                        Version
------------------------------ -----------
alembic                        1.18.1
anyio                          4.12.0
apimoex                        1.4.0
appnope                        0.1.4
argon2-cffi                    25.1.0
argon2-cffi-bindings           25.1.0
arrow                          1.4.0
asttokens                      3.0.0
async-generator                1.10
async-lru                      2.0.5
attrs                          25.3.0
babel                          2.17.0
beautifulsoup4                 4.14.3
bleach                         6.3.0
boto                           2.49.0
catboost                       1.2.8
certifi                        2026.1.4
cffi                           2.0.0
charset-normalizer             3.4.4
cloudpickle                    3.1.2
colorlog                       6.10.1
comm                           0.2.2
commonmark                     0.9.1
contourpy                      1.3.3
cryptography       